# Pipeline

> Run OCR + fix the markdown headings + describe images/figures for a single pdf file

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs, ocr_pdf
from mistocr.refine import add_img_descs, fix_hdgs
from pathlib import Path
from asyncio import Semaphore, gather, sleep
import os, json, shutil

In [ ]:
#| export
@delegates(add_img_descs)
async def pdf_to_md(
    pdf_path:str, # Path to input PDF file
    dst:str, # Destination directory for output markdown
    ocr_output:str=None, # Optional OCR output directory (defaults to pdf_path stem)
    model:str='claude-sonnet-4-5', # Model to use for heading fixes and image descriptions
    add_img_desc:bool=True, # Whether to add image descriptions
    progress:bool=True, # Whether to show progress messages
    **kwargs):
    "Convert PDF to markdown with OCR, fixed heading hierarchy, and optional image descriptions"
    ocr_dir = Path(ocr_output) if ocr_output else Path(pdf_path).with_suffix('')
    n_steps = 3 if add_img_desc else 2
    if progress: print(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_pdf(pdf_path, ocr_dir)
    if progress: print(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: print(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst and Path(dst) != ocr_dir:
        shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if progress: print("Done!")

In [ ]:
ocr_pdf?

Signature:
ocr_pdf(
    path: str,
    dst: str = 'md',
    inc_img: bool = True,
    key: str = None,
    poll_interval: int = 2,
) -> list[pathlib.Path]
Docstring: OCR a PDF file or folder of PDFs and save results
File:      ~/dev/mistocr/mistocr/core.py
Type:      function

In [ ]:
!ls files/test

attention-is-all-you-need.pdf  md  md_all  md_enriched	md_fixed  resnet.pdf


Can we test it, for instance using "attention-is-all-you-need.pdf" paper?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
await pdf_to_md('files/test/attention-is-all-you-need.pdf', 'files/test/md_test')
```

In [ ]:
await pdf_to_md('files/test/attention-is-all-you-need.pdf', 'files/test/md_test')

Step 1/3: Running OCR on files/test/attention-is-all-you-need.pdf...


Mistral batch job status: QUEUED


Mistral batch job status: RUNNING


Step 2/3: Fixing heading hierarchy...


Step 3/3: Adding image descriptions...


FileNotFoundError: [Errno 2] No such file or directory: 'files/test/attention-is-all-you-need/img'

Any idea why this error?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The error occurs because `add_img_descs` is trying to list files in an `img` folder that doesn't exist. The OCR process likely didn't extract any images from the PDF, so the `img` directory was never created.

Let's check what's actually in the OCR output directory:

```python
!ls files/test/attention-is-all-you-need/
```

In [ ]:
!ls files/test/

attention-is-all-you-need      md      md_enriched  md_test
attention-is-all-you-need.pdf  md_all  md_fixed     resnet.pdf


In [ ]:
!ls -R files/test/attention-is-all-you-need


files/test/attention-is-all-you-need:
attention-is-all-you-need

files/test/attention-is-all-you-need/attention-is-all-you-need:
img	    page_11.md	page_14.md  page_3.md  page_6.md  page_9.md
page_1.md   page_12.md	page_15.md  page_4.md  page_7.md
page_10.md  page_13.md	page_2.md   page_5.md  page_8.md

files/test/attention-is-all-you-need/attention-is-all-you-need/img:
img-0.jpeg  img-1.jpeg	img-2.jpeg  img-3.jpeg	img-4.jpeg


In [ ]:
!ls files/test/md_test